In [ ]:
!pip install -q peft transformers==4.46.0 accelerate flask pyngrok soundfile librosa numpy torch

In [ ]:
# Upload ton zip modèle ici
from google.colab import files
import zipfile, os

print("Upload ton fichier whisper-wolof-model-p2.zip...")
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
os.makedirs('/content/model', exist_ok=True)
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/model')

print("Modèle extrait !")
print(os.listdir('/content/model'))

In [ ]:
import torch
from transformers import WhisperForConditionalGeneration, WhisperProcessor, pipeline
from peft import PeftModel

MODEL_ID = "openai/whisper-large-v3"
ADAPTER_PATH = "/content/model"

print("Chargement du processeur...")
processor = WhisperProcessor.from_pretrained(ADAPTER_PATH)

print("Chargement de Whisper Large V3...")
base_model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)

print("Application du LoRA...")
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model = model.merge_and_unload()

print("Création du pipeline...")
pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    torch_dtype=torch.float16,
    device="cuda",
    chunk_length_s=30,
    batch_size=4,
)

print("\n=== MODÈLE PRÊT ! ===")

In [ ]:
# Lancer le serveur API
import threading
import tempfile
import json
import numpy as np
import soundfile as sf
import librosa
from flask import Flask, request, jsonify
from pyngrok import ngrok

app = Flask(__name__)

@app.route('/transcribe', methods=['POST'])
def transcribe():
    if 'file' not in request.files:
        return jsonify({'error': 'No file'}), 400

    file = request.files['file']
    with tempfile.NamedTemporaryFile(delete=False, suffix='.wav') as tmp:
        file.save(tmp.name)
        tmp_path = tmp.name

    try:
        result = pipe(
            tmp_path,
            return_timestamps=True,
            generate_kwargs={"language": "fr", "task": "transcribe"},
        )

        segments = []
        if result.get('chunks'):
            for chunk in result['chunks']:
                ts = chunk.get('timestamp', (0, 0))
                segments.append({
                    'start': round(ts[0] or 0, 2),
                    'end': round(ts[1] or 0, 2),
                    'text': chunk['text'].strip(),
                })

        return jsonify({
            'text': result['text'].strip(),
            'segments': segments,
            'language': 'wo',
        })
    finally:
        import os
        os.unlink(tmp_path)

@app.route('/health', methods=['GET'])
def health():
    return jsonify({'status': 'ok', 'model': 'whisper-wolof-finetuned'})

# Lancer ngrok + serveur
ngrok.kill()  # kill any existing tunnels

# Tu peux obtenir un token gratuit sur ngrok.com
# ngrok.set_auth_token("TON_TOKEN_NGROK")

public_url = ngrok.connect(5000)
print(f"\n{'='*60}")
print(f"   SERVEUR WOLOF ASR PRÊT !")
print(f"   URL publique : {public_url}")
print(f"   ")
print(f"   Copie cette URL dans ton .env :")
print(f"   WOLOF_API_URL={public_url}")
print(f"{'='*60}\n")
print("Endpoints:")
print(f"  POST {public_url}/transcribe  (envoie un fichier audio)")
print(f"  GET  {public_url}/health")
print("\nLe serveur tourne... ne ferme pas ce notebook !")

app.run(port=5000)